# 11 · Real-world survey evaluation

This notebook evaluates the saved CNN on labeled handwritten digits collected through the project survey. It groups results by writer, creates a prediction sheet showing every image, and creates a score graphic for each writer.

The survey has not been collected yet. When ready, place the exported archive at `data/survey/realworld_dataset.npz`. The archive should contain `X_test`, `Y_test`, and `writer` arrays, matching the provided example. The code also accepts `images`, `labels`, and `writer_name` as field names. Images should already be single grayscale digits resized and oriented to match the model's 28 × 28 input.


In [1]:
from pathlib import Path

import pandas as pd

from src.realworld_evaluation import evaluate_survey_file

SURVEY_FILE = Path("../data/survey/realworld_dataset.npz")
MODEL_FILE = Path("../models/cnn_writer_independent.keras")
OUTPUT_DIR = Path("../outputs/realworld")
PREDICTION_IMAGE_SCALE = 3


## Check the input files

Confirm the survey archive and saved model are present before evaluation. This makes a missing survey export or model file clear before running predictions.


In [2]:
if not SURVEY_FILE.is_file():
    raise FileNotFoundError(
        f"Survey file not found: {SURVEY_FILE}. "
        "After collecting the survey, place realworld_dataset.npz in data/survey/."
    )
if not MODEL_FILE.is_file():
    raise FileNotFoundError(
        f"Saved model not found: {MODEL_FILE}. Run notebook 08 and save its selected model first."
    )

print("Survey archive:", SURVEY_FILE)
print("Model:", MODEL_FILE)
print("Output folder:", OUTPUT_DIR)


Survey archive: ../data/survey/realworld_dataset.npz
Model: ../models/cnn_writer_independent.keras
Output folder: ../outputs/realworld


## Predict and create writer reports

The evaluation function loads the labeled images and saved model, predicts in batches, calculates accuracy and recall by writer, and writes two PNGs per writer into `outputs/realworld/`. Green labels indicate correct predictions; red labels indicate mistakes.


In [3]:
results = evaluate_survey_file(
    npz_path=SURVEY_FILE,
    model_path=MODEL_FILE,
    output_dir=OUTPUT_DIR,
    batch_size=128,
    prediction_scale=PREDICTION_IMAGE_SCALE,
)


I0000 00:00:1790219436.437042  140048 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1790219436.478259  140048 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1790219437.496880  140048 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
W0000 00:00:1790219439.888694  140048 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries men

10000: accuracy=93.00%, macro recall=93.00%, correct=93, wrong=7
6969: accuracy=93.00%, macro recall=93.00%, correct=93, wrong=7


## Review the scores by writer

Accuracy is the share of all that writer's images predicted correctly. Macro recall is the average recall across digits present in that writer's labels. Correct and wrong counts show the number of images behind the score; per-digit recall reveals which handwritten digits were easiest or hardest for the model.


In [4]:
writer_rows = []
for writer, metrics in results["writer_metrics"].items():
    writer_rows.append({
        "writer": writer,
        "images": metrics["total"],
        "correct": metrics["correct"],
        "wrong": metrics["wrong"],
        "accuracy": metrics["accuracy"],
        "macro_recall": metrics["macro_recall"],
    })

writer_summary = pd.DataFrame(writer_rows).set_index("writer")
formatted_summary = writer_summary.copy()
formatted_summary["accuracy"] = formatted_summary["accuracy"].map(lambda value: f"{value:.2%}")
formatted_summary["macro_recall"] = formatted_summary["macro_recall"].map(lambda value: f"{value:.2%}")
display(formatted_summary)


,images,correct,wrong,accuracy,macro_recall
writer,,,,,
10000,100,93,7,93.00%,93.00%
6969,100,93,7,93.00%,93.00%


## Locate the generated PNG reports

The evaluation saves one prediction sheet and one score chart per writer in the output folder. Set `PREDICTION_IMAGE_SCALE` in Cell 1 to an integer `k`; it scales the canvas, enlarged image pixels, text, gaps, and margins together. For 100 images and the current 200-pixel header, the prediction PNG is **2,910k × 3,310k pixels** (for example, scale 2 gives 5,820 × 6,620). The prediction sheet has a dark background, centered writer/results header, and images filled top-to-bottom within each column. Captions below each image show prediction, true label, and confidence. The score chart shows accuracy, macro recall, correct/wrong counts, and recall by digit. This cell lists the file paths without embedding every writer’s image in the notebook output.


In [5]:
for writer in results["writer_metrics"]:
    prediction_path = results["prediction_sheets"][writer]
    metrics_path = results["metric_charts"][writer]
    print(f"Writer: {writer}")
    print("  Prediction sheet:", prediction_path)
    print("  Score chart:     ", metrics_path)


Writer: 10000
  Prediction sheet: ../outputs/realworld/10000_prediction.png
  Score chart:      ../outputs/realworld/10000_metrics.png
Writer: 6969
  Prediction sheet: ../outputs/realworld/6969_prediction.png
  Score chart:      ../outputs/realworld/6969_metrics.png


## Results and interpretation

Review each writer's score chart and prediction sheet. Report the number of survey images, correct and incorrect predictions, accuracy, macro recall, and the digits with the lowest recall. Keep each writer's results separate so participants can see how the model performed on their own handwriting.

These measurements describe the survey submissions only. They are not a replacement for the held-out EMNIST evaluation, and they should not be used to tune the saved model if the survey is being treated as a final real-world check.
